In [1]:
!pip install -q -U transformers accelerate bitsandbytes datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.


In [2]:
# ============ CONFIG ============
MODEL_NAME  = "Qwen/Qwen2.5-Coder-14B-Instruct"   
USE_4BIT    = True    
N_PROBLEMS  = 164         
MAX_ITER    = 3
FEEDBACK    = "generated"  
REFORMULATE = True      
OUT_PATH    = "/kaggle/working/results_builder_generated.json"

In [3]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
kw = dict(device_map="auto")
if USE_4BIT:
    kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True,
                                                   bnb_4bit_compute_dtype=torch.float16)
else:
    kw["torch_dtype"] = torch.float16      # T4 ne supporte pas bfloat16
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kw).eval()
print("Modèle chargé sur :", model.device)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/47.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Modèle chargé sur : cpu


In [4]:
import re, sys, os, json, time, tempfile, subprocess
from collections import Counter

class LLM:
    def __init__(self):
        self.tokens = 0
    def chat(self, system, user, max_tokens=1024):
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        gen = out[0][inputs.input_ids.shape[1]:]
        self.tokens += inputs.input_ids.shape[1] + len(gen)
        return tok.decode(gen, skip_special_tokens=True)

ROUTER_SYS = (
    "Classify the user's message. Respond only with "
    "'CODE' if the message is a code generation, code correction, "
    "or code modification request. Otherwise, respond with 'OTHER'."
)
REFORM_SYS = (
    "You are an expert in prompt engineering and software requirements analysis. "
    "Your task is to transform the user's request into a clear, precise, "
    "and technically structured prompt for code generation.\n\n"

    "Before generating the final prompt, carefully analyze the user's request "
    "and identify the following elements:\n"
    "1. Objective: What exactly should the code accomplish?\n"
    "2. Programming language: Specify the required language (Python by default).\n"
    "3. Inputs: Identify all input parameters, their data types, and expected formats.\n"
    "4. Outputs: Specify the expected return value, its data type, and format.\n"
    "5. Function signature: Define the exact function name, parameters, and return type "
    "whenever applicable.\n"
    "6. Conditions and logic: Describe the rules, operations, and conditions "
    "that the code must follow.\n"
    "7. Constraints: Specify technical requirements, restrictions, and implementation rules.\n"
    "8. Edge cases: Identify exceptional situations and boundary cases that must be handled.\n"
    "9. Error handling: Specify how invalid inputs or unexpected situations should be handled.\n\n"

    "If the user's request is ambiguous or lacks essential information, "
    "do not invent requirements. Explicitly identify what needs clarification.\n\n"

    "Organize the reformulated prompt into a structured technical specification "
    "that a Python developer or code generation model can directly implement.\n\n"

    "Do not generate code. Do not provide explanations about your analysis. "
    "Output ONLY the final reformulated prompt."
)
    
BUILDER_SYS = (
    "You are an expert Python software engineer. "
    "Your task is to implement the technical specification provided to you.\n\n"

    "Strict requirements:\n"
    "- Follow all specified inputs, outputs, conditions, and constraints.\n"
    "- Respect the exact function signature defined in the specification.\n"
    "- Handle all specified edge cases and errors.\n"
    "- Write clean, readable, efficient, and well-structured Python code.\n"
    "- Include all necessary imports.\n"
    "- Implement the complete solution without omitting any requirements.\n\n"

    "Output requirements:\n"
    "- Return exactly ONE ```python code block.\n"
    "- Include imports and the complete function implementation.\n"
    "- Do not include explanations, comments outside the code, "
    "usage examples, or input() statements."
)
def is_code_request(llm, msg):
    return llm.chat(ROUTER_SYS, msg, max_tokens=5).strip().upper().startswith("CODE")

def reformulate(llm, msg):
    return llm.chat(REFORM_SYS, msg, max_tokens=512).strip()

def extract_code(text):
    m = re.findall(r"```(?:python)?\n(.*?)```", text, re.S)
    return (m[0] if m else text).strip()

def run_sandbox(code, extra="", timeout=10):
    try:
        compile(code, "<gen>", "exec")
    except SyntaxError as e:
        return False, "SyntaxError", f"{e.msg} (ligne {e.lineno})"
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "sol.py")
        with open(path, "w") as f:
            f.write(code + "\n\n" + extra)
        try:
            p = subprocess.run([sys.executable, path], capture_output=True,
                               text=True, timeout=timeout, cwd=d)
        except subprocess.TimeoutExpired:
            return False, "Timeout", f"Execution > {timeout}s"
    if p.returncode == 0:
        return True, None, ""
    lines = p.stderr.strip().splitlines()
    last = lines[-1] if lines else "unknown"
    etype = "AssertionError" if "AssertionError" in p.stderr else (last.split(":")[0] if ":" in last else "RuntimeError")
    return False, etype, p.stderr[-1500:]

def solve(llm, message, problem, max_iter, feedback, reformulate_on):
    t0, trace = time.time(), []
    if not is_code_request(llm, message):
        return {"intent": "OTHER", "success": None, "iters": 0, "trace": [], "latency": time.time() - t0}
    prompt = reformulate(llm, message) if reformulate_on else message
    loop_extra = problem["test"] + f"\ncheck({problem['entry_point']})\n" if feedback == "tests" else ""
    user_msg, code = prompt, ""
    for i in range(1, max_iter + 1):
        code = extract_code(llm.chat(BUILDER_SYS, user_msg, max_tokens=768))
        ok, etype, err = run_sandbox(code, loop_extra)
        trace.append({"iter": i, "ok": ok, "error": etype})
        if ok:
            break
        user_msg = (f"{prompt}\n\nTon code precedent :\n```python\n{code}\n```\n"
                    f"a echoue avec {etype} :\n{err}\nCorrige-le. Renvoie le code complet.")
    final_extra = problem["test"] + f"\ncheck({problem['entry_point']})\n"
    final_ok, final_err, _ = run_sandbox(code, final_extra)
    return {"intent": "CODE", "success": final_ok, "final_error": final_err, "iters": len(trace),
            "first_try_ok": final_ok and len(trace) == 1, "trace": trace, "latency": time.time() - t0}

def compute_metrics(results, llm):
    code_res = [r for r in results if r["intent"] == "CODE"]
    n = len(code_res)
    succ = [r for r in code_res if r["success"]]
    first = [r for r in code_res if r.get("first_try_ok")]
    errs = Counter(t["error"] for r in code_res for t in r["trace"] if not t["ok"])
    return {
        "n_problems": n,
        "intent_accuracy_on_code": n / max(len(results), 1),
        "pass@1": len(first) / max(n, 1),
        "pass@final": len(succ) / max(n, 1),
        "gain_reparation": (len(succ) - len(first)) / max(n, 1),
        "avg_iterations": sum(r["iters"] for r in code_res) / max(n, 1),
        "avg_latency_s": sum(r["latency"] for r in results) / max(len(results), 1),
        "tokens_per_problem": llm.tokens / max(n, 1),
        "error_distribution": dict(errs),
        "success_by_iteration": {k: sum(1 for r in succ if r["iters"] == k) for k in range(1, 6)},
    }

## Mode `generated` : tests générés par le modèle (sans fuite)
Remplace `solve` : le feedback vient des exemples du docstring + de tests écrits par le modèle à partir de l'énoncé.

In [5]:
import ast, doctest

# ---------- 1) Génération de tests par le modèle (sans fuite : seulement à partir de l'énoncé) ----------
TESTGEN_SYS = ("You are an expert in Python unit testing. Based on the problem statement, write 6 to 8 "
               "`assert` statements for the requested function, with EACH statement on a SINGLE line, "
               "using the format `assert function_name(args) == expected_value`. Include both normal "
               "cases AND edge cases (empty input, zero, negative value, single element, duplicates). "
               "Do NOT implement the function. Respond with ONLY ONE ```python code block containing "
               "these assert statements and nothing else.")

def generate_tests(llm, message, entry_point, max_tests=8):
    raw = llm.chat(TESTGEN_SYS, f"Fonction : {entry_point}\n\nEnonce :\n{message}", max_tokens=400)
    lines = []
    for l in extract_code(raw).splitlines():
        l = l.strip()
        if not l.startswith("assert") or entry_point not in l:
            continue
        try:
            if isinstance(ast.parse(l).body[0], ast.Assert):
                lines.append(l)
        except SyntaxError:
            pass
    return lines[:max_tests]

# ---------- 2) Construction du code de vérification (chaque échec est collecté, pas seulement le 1er) ----------
CHECK_HEAD = "\n_fails = []\n"
CHECK_TAIL = "assert not _fails, 'Echecs : ' + ' ; '.join(str(x) for x in _fails)\n"

def example_checks(prompt):
    # Exemples '>>> f(x)' du docstring : déjà visibles dans le prompt, donc pas de fuite
    body = ""
    for ex in doctest.DocTestParser().get_examples(prompt):
        src = ex.source.strip()
        want_lines = ex.want.strip().splitlines()
        # le parseur colle parfois la fin du docstring a la sortie attendue : on ne garde que la 1re ligne
        if not want_lines or want_lines[0].strip().startswith(('"' * 3, "'" * 3)):
            continue
        want = want_lines[0].strip()
        try:
            ast.parse(src, mode="eval")
        except SyntaxError:
            continue
        body += (f"try:\n"
                 f"    _g = eval({src!r})\n"
                 f"    if repr(_g) != {want!r}:\n"
                 f"        _fails.append(({src!r}, 'obtenu: ' + repr(_g) + ' | attendu: ' + {want!r}))\n"
                 f"except Exception as _e:\n"
                 f"    _fails.append(({src!r}, type(_e).__name__ + ': ' + str(_e)))\n")
    return body

def generated_checks(lines):
    body = ""
    for line in lines:
        t = ast.parse(line).body[0].test
        if isinstance(t, ast.Compare) and len(t.ops) == 1 and isinstance(t.ops[0], ast.Eq):
            left, right = ast.unparse(t.left), ast.unparse(t.comparators[0])
            body += (f"try:\n"
                     f"    _got = {left}\n"
                     f"    _exp = {right}\n"
                     f"    if _got != _exp:\n"
                     f"        _fails.append(({line!r}, 'obtenu: ' + repr(_got) + ' | attendu: ' + repr(_exp)))\n"
                     f"except Exception as _e:\n"
                     f"    _fails.append(({line!r}, type(_e).__name__ + ': ' + str(_e)))\n")
        else:
            body += (f"try:\n    {line}\n"
                     f"except Exception as _e:\n"
                     f"    _fails.append(({line!r}, type(_e).__name__ + ': ' + str(_e)))\n")
    return body

def build_loop_checker(*bodies):
    body = "".join(bodies)
    return CHECK_HEAD + body + CHECK_TAIL if body else ""

# ---------- 3) solve() : nouveaux modes "examples" et "generated" ----------
def solve(llm, message, problem, max_iter, feedback, reformulate_on):
    t0, trace = time.time(), []
    if not is_code_request(llm, message):
        return {"intent": "OTHER", "success": None, "iters": 0, "trace": [], "latency": time.time() - t0}

    prompt = reformulate(llm, message) if reformulate_on else message
    hidden = problem["test"] + f"\ncheck({problem['entry_point']})\n"   # jamais montré au modèle (sauf mode "tests")

    loop_extra, n_gen = "", 0
    if feedback == "tests":
        loop_extra = hidden
    elif feedback == "examples":
        loop_extra = build_loop_checker(example_checks(problem["prompt"]))
    elif feedback == "generated":
        gen_lines = generate_tests(llm, message, problem["entry_point"])
        n_gen = len(gen_lines)
        loop_extra = build_loop_checker(example_checks(problem["prompt"]), generated_checks(gen_lines))

    user_msg, code = prompt, ""
    for i in range(1, max_iter + 1):
        code = extract_code(llm.chat(BUILDER_SYS, user_msg, max_tokens=768))
        ok, etype, err = run_sandbox(code, loop_extra)                    # feedback visible par le modèle
        h_ok, h_err, _ = run_sandbox(code, hidden)                         # mesure uniquement
        trace.append({"iter": i, "ok": ok, "error": etype, "hidden_ok": h_ok})
        if ok:
            break
        user_msg = (f"{prompt}\n\nTon code precedent :\n```python\n{code}\n```\n"
                    f"a echoue avec {etype} :\n{err}\n"
                    "Remarque : les tests peuvent etre approximatifs ; s'ils contredisent l'enonce, "
                    "fais confiance a l'enonce.\nCorrige-le. Renvoie le code complet.")

    return {"intent": "CODE", "success": h_ok, "final_error": h_err, "iters": len(trace),
            "first_try_ok": trace[0]["hidden_ok"],          # vrai pass@1 : 1er code évalué sur les tests cachés
            "n_gen_tests": n_gen, "trace": trace, "latency": time.time() - t0}

In [ ]:
from datasets import load_dataset
ds = load_dataset("openai/openai_humaneval", split="test").select(range(N_PROBLEMS))

# Reprise automatique si la session a été coupée
results, done = [], set()
if os.path.exists(OUT_PATH):
    results = json.load(open(OUT_PATH))["results"]
    done = {r["task_id"] for r in results}
    print(f"Reprise : {len(done)} problèmes déjà faits")

llm = LLM()
for k, p in enumerate(ds):
    if p["task_id"] in done:
        continue
    r = solve(llm, p["prompt"], p, MAX_ITER, FEEDBACK, REFORMULATE)
    r["task_id"] = p["task_id"]
    results.append(r)
    print(f"[{k+1}/{len(ds)}] {p['task_id']}  success={r['success']}  iters={r['iters']}  {r['latency']:.0f}s")
    json.dump({"config": dict(MODEL_NAME=MODEL_NAME, MAX_ITER=MAX_ITER, FEEDBACK=FEEDBACK,
                              REFORMULATE=REFORMULATE), "results": results},
              open(OUT_PATH, "w"), ensure_ascii=False)

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

code_res = [r for r in results if r["intent"] == "CODE"]
n = len(code_res)
m = compute_metrics(results, llm)

fig, ax = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f"Stratégie 2 (Builder seul) | {MODEL_NAME.split('/')[-1]} | {n} problèmes HumanEval",
             fontsize=15, fontweight="bold")

# 1) pass@1 vs pass@final
bars = ax[0, 0].bar(["pass@1", "pass@final"], [m["pass@1"] * 100, m["pass@final"] * 100],
                    color=["#9ecae1", "#2171b5"])
ax[0, 0].bar_label(bars, fmt="%.1f%%", fontsize=12)
ax[0, 0].set_ylim(0, 105); ax[0, 0].set_ylabel("% de problèmes résolus")
ax[0, 0].set_title("Précision : avant / après réparation")

# 2) Résolus cumulés par itération
ks = list(range(1, MAX_ITER + 1))
solved = [sum(1 for r in code_res if r["success"] and r["iters"] == k) for k in ks]
cum = [sum(1 for r in code_res if r["trace"][min(k, len(r["trace"])) - 1]["hidden_ok"]) / max(n, 1) * 100 for k in ks]
ax[0, 1].plot(ks, cum, marker="o", linewidth=2, color="#2171b5")
for x, y in zip(ks, cum):
    ax[0, 1].annotate(f"{y:.1f}%", (x, y), textcoords="offset points", xytext=(0, 8), ha="center")
ax[0, 1].set_xticks(ks); ax[0, 1].set_ylim(0, 105)
ax[0, 1].set_xlabel("Itération"); ax[0, 1].set_ylabel("% résolus (cumulé)")
ax[0, 1].set_title("Gain apporté par chaque itération")

# 3) Types d'erreurs des échecs finaux
fails = Counter((r.get("final_error") or "Inconnu") for r in code_res if not r["success"])
if fails:
    ax[0, 2].barh(list(fails.keys()), list(fails.values()), color="#de2d26")
    ax[0, 2].bar_label(ax[0, 2].containers[0])
else:
    ax[0, 2].text(0.5, 0.5, "Aucun échec", ha="center", va="center", fontsize=14)
ax[0, 2].set_title("Types d'erreurs (échecs finaux)"); ax[0, 2].set_xlabel("Nombre de problèmes")

# 4) Distribution du nombre d'itérations utilisées
it = Counter(r["iters"] for r in code_res)
ax[1, 0].bar([str(k) for k in ks], [it.get(k, 0) for k in ks], color="#31a354")
ax[1, 0].bar_label(ax[1, 0].containers[0])
ax[1, 0].set_xlabel("Itérations utilisées"); ax[1, 0].set_ylabel("Nombre de problèmes")
ax[1, 0].set_title(f"Itérations par problème (moy. {m['avg_iterations']:.2f})")

# 5) Latence par problème
lat = [r["latency"] for r in results]
ax[1, 1].hist(lat, bins=20, color="#fd8d3c", edgecolor="white")
ax[1, 1].axvline(np.mean(lat), color="k", linestyle="--", label=f"moy. {np.mean(lat):.1f}s")
ax[1, 1].set_xlabel("Latence (s)"); ax[1, 1].set_ylabel("Nombre de problèmes")
ax[1, 1].set_title("Distribution de la latence"); ax[1, 1].legend()

# 6) Tableau récapitulatif
rows = [["Problèmes (code)", f"{n}"],
        ["pass@1", f"{m['pass@1'] * 100:.1f}%"],
        ["pass@final", f"{m['pass@final'] * 100:.1f}%"],
        ["Gain réparation", f"+{m['gain_reparation'] * 100:.1f} pts"],
        ["Itérations moy.", f"{m['avg_iterations']:.2f}"],
        ["Latence moy.", f"{m['avg_latency_s']:.1f}s"],
        ["Tokens / problème", f"{m['tokens_per_problem']:.0f}"],
        ["Feedback / Reformul.", f"{FEEDBACK} / {REFORMULATE}"]]
ax[1, 2].axis("off")
tbl = ax[1, 2].table(cellText=rows, colLabels=["Métrique", "Valeur"], loc="center", cellLoc="left")
tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1, 1.8)
ax[1, 2].set_title("Résumé")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("/kaggle/working/performance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import json
from collections import Counter

d = json.load(open("/kaggle/working/results_builder_generated.json"))["results"]
code_res = [r for r in d if r["intent"] == "CODE"]
t1 = lambda r: r["trace"][0]

fixed       = [r for r in code_res if not t1(r)["hidden_ok"] and r["success"]]       # corrigés par la boucle
broken      = [r for r in code_res if t1(r)["hidden_ok"] and not r["success"]]       # code juste abîmé par la boucle
false_alarm = [r for r in code_res if t1(r)["hidden_ok"] and not t1(r)["ok"]]        # code juste, rejeté par les tests générés
caught      = [r for r in code_res if not t1(r)["hidden_ok"] and not t1(r)["ok"]]    # code faux, détecté
missed      = [r for r in code_res if not t1(r)["hidden_ok"] and t1(r)["ok"]]        # code faux, non détecté

n_ok1 = sum(t1(r)["hidden_ok"] for r in code_res)
print(f"Corrigés par la boucle   : {len(fixed)}")
print(f"Abîmés par la boucle     : {len(broken)}")
print(f"Fausses alertes          : {len(false_alarm)} / {n_ok1} codes justes ({len(false_alarm)/n_ok1:.0%})")
print(f"Vrais échecs détectés    : {len(caught)} / {len(caught)+len(missed)}")
print("Tests valides par problème :", dict(Counter(r["n_gen_tests"] for r in code_res)))

In [ ]:
pip install bigcodebench


In [ ]:
from dataclasses import dataclass, field

@dataclass
class Task:
    task_id: str
    dataset: str
    prompt: str
    entry_point: str
    trusted_tests: list = field(default_factory=list)
    extra: dict = field(default_factory=dict)   

In [ ]:
# ======================================== BIGCODEBENCH — MÉTRIQUES DÉTAILLÉES ========================================
import json, os, re, random, time, ast
from collections import Counter
from dataclasses import dataclass, field
import numpy as np
import matplotlib.pyplot as plt

# ----- CONFIG -----
SUBSET         = "hard"
PROMPT_SPLIT   = "instruct"
EXECUTION      = "gradio"
N_TASKS        = 50               # ← augmenté (20 → 50)
MAX_ITER       = 3
FEEDBACK       = False
REFORMULATE    = False
BCB_OUT        = "/kaggle/working/results_bcb.json"
WORK_DIR       = "/kaggle/working/bcb_work"

# ----- Task -----
@dataclass
class Task:
    task_id: str
    dataset: str
    prompt: str
    entry_point: str
    trusted_tests: list = field(default_factory=list)
    extra: dict = field(default_factory=dict)

# ----- Helpers -----
def extract_code(text: str) -> str:
    blocks = re.findall(r"```(?:python|py)?[ \t]*\n(.*?)```", text, re.S)
    if blocks:
        return max(blocks, key=len).strip()
    m = re.search(r"```(?:python|py)?[ \t]*\n(.*)", text, re.S)
    if m:
        return m.group(1).strip()
    return text.strip()

def syntax_error(code: str) -> str | None:
    try:
        ast.parse(code)
        return None
    except SyntaxError as e:
        return f"SyntaxError: {e.msg} (line {e.lineno})"

def has_infra_error(r: dict) -> bool:
    return r.get("intent") == "CODE" and r.get("final_error") == "SandboxInfraError"

# ----- solve -----
BUILDER_SYS = "Tu es un expert en programmation Python. Renvoie uniquement le code complet dans un bloc ```python."

def solve(llm, message, problem, max_iter, feedback, reformulate_on):
    t0 = time.time()
    trace = []
    prompt = reformulate(llm, message) if reformulate_on else message
    user_msg, code = prompt, ""
    for i in range(1, max_iter + 1):
        code = extract_code(llm.chat(BUILDER_SYS, user_msg, max_tokens=2048))
        err = syntax_error(code)
        trace.append({"iter": i, "ok": err is None, "error": err})
        if err is None:
            break
        user_msg = (f"{prompt}\n\nTon code précédent :\n```python\n{code}\n```\n"
                    f"a échoué avec {err}\nCorrige-le. Renvoie le code complet.")
    return {
        "intent": "CODE", "success": None, "final_code": code,
        "final_error": syntax_error(code), "iters": len(trace),
        "first_try_ok": len(trace) == 1 and syntax_error(code) is None,
        "trace": trace, "latency": time.time() - t0,
    }

# ----- 1) Chargement -----
def load_tasks(subset=SUBSET, split=PROMPT_SPLIT):
    from bigcodebench.data import get_bigcodebench
    key = "complete_prompt" if split == "complete" else "instruct_prompt"
    tasks = [Task(task_id=tid, dataset="bigcodebench", prompt=p[key],
                  entry_point=p.get("entry_point", "task_func"),
                  trusted_tests=[], extra={"libs": p.get("libs")})
             for tid, p in get_bigcodebench(subset=subset).items()]
    return sorted(tasks, key=lambda t: int(t.task_id.split("/")[1]))

# ----- 2) Scoring via le Space -----
def score(solutions: dict, work_dir: str = WORK_DIR, calibrated=True) -> dict:
    from bigcodebench.evaluate import evaluate
    from bigcodebench.data import get_bigcodebench
    os.makedirs(work_dir, exist_ok=True)
    samples_path = os.path.join(work_dir, "samples.jsonl")
    results_path = samples_path.replace(".jsonl", "_eval_results.json")

    def write_samples(items):
        with open(samples_path, "w") as f:
            for tid, code in items.items():
                f.write(json.dumps({"task_id": tid, "solution": code or "pass"}) + "\n")
        if os.path.isfile(results_path):
            os.remove(results_path)

    write_samples(solutions)
    try:
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples_path,
                 execution=EXECUTION, calibrated=calibrated,
                 selective_evaluate=",".join(str(int(t.split("/")[1])) for t in solutions))
    except Exception as e:
        print(f"selective_evaluate failed → retry full")
        all_ids = list(get_bigcodebench(subset=SUBSET))
        write_samples({tid: solutions.get(tid, "pass") for tid in all_ids})
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples_path,
                 execution=EXECUTION, calibrated=calibrated)

    if not os.path.isfile(results_path):
        raise RuntimeError(f"Pas de fichier résultats à {results_path}")
    data = json.load(open(results_path))["eval"]
    return {tid: bool(data.get(tid)) and data[tid][0]["status"] == "pass" for tid in solutions}

# ----- 3) Évaluation -----
def evaluate_bcb(llm, tasks, max_iter=3, feedback=False, reformulate=False):
    state = json.load(open(BCB_OUT)) if os.path.exists(BCB_OUT) else {}
    results = state.get("results", [])
    done = {r["task_id"] for r in results}

    for k, task in enumerate(tasks):
        if task.task_id in done:
            continue
        r = solve(llm, task.prompt, {"test": "", "entry_point": task.entry_point},
                  max_iter, feedback, reformulate)
        if has_infra_error(r):
            raise RuntimeError("Panne sandbox : " + str(r.get("trace")))
        r["task_id"] = task.task_id
        results.append(r)
        n_ok = sum(bool(x["success"]) for x in results if x["success"] is not None)
        print(f"[{k+1}/{len(tasks)}] {task.task_id}  iters={r['iters']}  "
              f"{r['latency']:.0f}s  | cumul : {n_ok}/{len(results)}")
        json.dump({"config": {"subset": SUBSET, "split": PROMPT_SPLIT, "max_iter": max_iter},
                   "results": results}, open(BCB_OUT, "w"), ensure_ascii=False)
    return results

# ----- 4) MÉTRIQUES DÉTAILLÉES -----
def compute_metrics(results: list) -> dict:
    """Calcule toutes les métriques du builder."""
    scored = [r for r in results if r["success"] is not None]
    n = len(scored)
    if n == 0:
        return {}

    n_ok = sum(r["success"] for r in scored)
    n_first = sum(r.get("first_try_ok", False) for r in scored)
    n_syntax_err = sum(r.get("final_error") is not None for r in scored if r.get("final_error"))
    latencies = [r["latency"] for r in scored]
    iters = [r["iters"] for r in scored]

    return {
        "n_tasks": n,
        "pass@1": n_ok / n,
        "pass@first_try": n_first / n,
        "pass@final": n_ok / n,
        "avg_iterations": np.mean(iters),
        "max_iterations": max(iters),
        "avg_latency_s": np.mean(latencies),
        "median_latency_s": np.median(latencies),
        "p95_latency_s": np.percentile(latencies, 95),
        "n_syntax_errors": n_syntax_err,
        "syntax_error_rate": n_syntax_err / n,
        "n_one_iter": sum(1 for i in iters if i == 1),
        "n_multi_iter": sum(1 for i in iters if i > 1),
        "multi_iter_rate": sum(1 for i in iters if i > 1) / n,
    }

# ----- 5) VISUALISATION COMPLÈTE (6 panneaux) -----
def plot_bcb(results: list, max_iter: int = 3, save_path: str = None):
    scored = [r for r in results if r["success"] is not None]
    n = len(scored)
    if n == 0:
        print("Pas de résultats scorés.")
        return

    m = compute_metrics(results)

    fig, ax = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle(f"BigCodeBench ({SUBSET}) — {n} tâches — pass@1 = {m['pass@1']*100:.1f}%",
                 fontsize=14, fontweight="bold")

    # --- (1) pass@1 vs pass@first_try ---
    ax[0,0].bar(["pass@1\n(final)", "pass@first_try"],
                [m["pass@1"]*100, m["pass@first_try"]*100],
                color=["#2171b5", "#6baed6"], width=0.5)
    ax[0,0].bar_label(ax[0,0].containers[0], fmt="%.1f%%", fontsize=12)
    ax[0,0].set_ylim(0, 105)
    ax[0,0].set_title("Taux de succès")

    # --- (2) Distribution des itérations ---
    iters = Counter(r["iters"] for r in scored)
    ks = list(range(1, max_iter + 1))
    colors = ["#31a354" if k == 1 else "#74c476" if k == 2 else "#fdae61" for k in ks]
    ax[0,1].bar([str(k) for k in ks], [iters.get(k, 0) for k in ks], color=colors)
    ax[0,1].bar_label(ax[0,1].containers[0], fontsize=11)
    ax[0,1].set_xlabel("Nombre d'itérations")
    ax[0,1].set_title(f"Itérations (moy. {m['avg_iterations']:.2f})")
    ax[0,1].annotate(f"1er essai : {m['n_one_iter']}/{n} ({m['n_one_iter']/n*100:.0f}%)",
                     xy=(0.05, 0.95), xycoords="axes fraction", fontsize=10,
                     bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

    # --- (3) Types d'erreurs (échecs) ---
    fails = scored if False else [r for r in scored if not r["success"]]
    if fails:
        # Classer par type
        err_types = []
        for r in fails:
            fe = r.get("final_error")
            if fe and "SyntaxError" in fe:
                err_types.append("SyntaxError")
            elif fe and "NameError" in fe:
                err_types.append("NameError")
            elif fe and "ImportError" in fe:
                err_types.append("ImportError")
            elif fe and "TypeError" in fe:
                err_types.append("TypeError")
            elif fe:
                err_types.append(fe.split(":")[0].strip()[:20])
            else:
                err_types.append("TestFailure")
        ec = Counter(err_types)
        items = ec.most_common(10)
        ax[0,2].barh([x[0] for x in items][::-1], [x[1] for x in items][::-1], color="#de2d26")
        ax[0,2].bar_label(ax[0,2].containers[0], fontsize=10)
    else:
        ax[0,2].text(0.5, 0.5, "Aucun échec", ha="center", va="center", fontsize=14)
    ax[0,2].set_title(f"Types d'erreurs ({len(fails)} échecs)")

    # --- (4) Latence par tâche (boxplot + points) ---
    latencies = [r["latency"] for r in scored]
    bp = ax[1,0].boxplot(latencies, patch_artist=True, boxprops=dict(facecolor="#a1dab4"))
    ax[1,0].set_ylabel("Latence (s)")
    ax[1,0].set_title(f"Latence (méd. {m['median_latency_s']:.0f}s, p95 {m['p95_latency_s']:.0f}s)")
    ax[1,0].annotate(f"moy. {m['avg_latency_s']:.0f}s", xy=(1.3, m["avg_latency_s"]),
                     fontsize=10, color="red")

    # --- (5) Cumul du score (courbe) ---
    cum = 0
    cum_vals = []
    for r in scored:
        cum += 1 if r["success"] else 0
        cum_vals.append(cum)
    ax[1,1].plot(range(1, n+1), cum_vals, "o-", color="#2171b5", markersize=4, linewidth=1.5)
    ax[1,1].axhline(m["pass@1"]*n, color="gray", linestyle="--", alpha=0.5,
                    label=f"moy. = {m['pass@1']*n:.1f}")
    ax[1,1].set_xlabel("Tâche")
    ax[1,1].set_ylabel("Score cumulé")
    ax[1,1].set_title("Progression du score")
    ax[1,1].legend()

    # --- (6) Tableau récapitulatif ---
    ax[1,2].axis("off")
    table_data = [
        ["Métrique", "Valeur"],
        ["Tâches testées", str(n)],
        ["pass@1 (final)", f"{m['pass@1']*100:.1f}%"],
        ["pass@first_try", f"{m['pass@first_try']*100:.1f}%"],
        ["Itérations moy.", f"{m['avg_iterations']:.2f}"],
        ["Multi-itérations", f"{m['n_multi_iter']}/{n} ({m['multi_iter_rate']*100:.0f}%)"],
        ["Latence moy.", f"{m['avg_latency_s']:.1f}s"],
        ["Latence médiane", f"{m['median_latency_s']:.1f}s"],
        ["Latence p95", f"{m['p95_latency_s']:.1f}s"],
        ["Erreurs de syntaxe", f"{m['n_syntax_errors']} ({m['syntax_error_rate']*100:.0f}%)"],
    ]
    table = ax[1,2].table(cellText=table_data[1:], colLabels=table_data[0],
                          loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.5)
    for i in range(len(table_data)):
        table[0, i].set_facecolor("#4393c3")
        table[0, i].set_text_props(color="white", fontweight="bold")
        if i > 0 and i % 2 == 0:
            for j in range(2):
                table[i, j].set_facecolor("#f0f0f0")
    ax[1,2].set_title("Récapitulatif")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    # --- Affichage console ---
    print("\n" + "="*50)
    print(f"  BigCodeBench ({SUBSET}) — {n} tâches")
    print("="*50)
    print(f"  pass@1 (final)      : {m['pass@1']*100:.1f}%  ({sum(r['success'] for r in scored)}/{n})")
    print(f"  pass@first_try      : {m['pass@first_try']*100:.1f}%  ({sum(r.get('first_try_ok', False) for r in scored)}/{n})")
    print(f"  Itérations moy.     : {m['avg_iterations']:.2f}")
    print(f"  Multi-itérations    : {m['n_multi_iter']}/{n} ({m['multi_iter_rate']*100:.0f}%)")
    print(f"  Latence moy.        : {m['avg_latency_s']:.1f}s")
    print(f"  Latence médiane     : {m['median_latency_s']:.1f}s")
    print(f"  Latence p95         : {m['p95_latency_s']:.1f}s")
    print(f"  Erreurs syntaxe     : {m['n_syntax_errors']} ({m['syntax_error_rate']*100:.0f}%)")
    print("="*50)

    # --- Détail par tâche ---
    print(f"\n{'Tâche':<22} {'OK':<4} {'Iters':<6} {'Lat(s)':<8} {'Erreur'}")
    print("-"*70)
    for r in scored:
        status = "✓" if r["success"] else "✗"
        err = (r.get("final_error") or "")[:40]
        print(f"{r['task_id']:<22} {status:<4} {r['iters']:<6} {r['latency']:<8.1f} {err}")

# ----- 6) Lancement -----
ALL_TASKS = load_tasks()
random.Random(0).shuffle(ALL_TASKS)
TASKS = ALL_TASKS[:N_TASKS]
print(f"{len(ALL_TASKS)} tâches au total, {len(TASKS)} sélectionnées")

llm = LLM()
results = evaluate_bcb(llm, TASKS, MAX_ITER, FEEDBACK, REFORMULATE)

# Scoring final via le Space
solutions = {r["task_id"]: r["final_code"] for r in results}
scores = score(solutions)
for r in results:
    r["success"] = scores.get(r["task_id"], False)

# Sauvegarde
json.dump({"config": {"subset": SUBSET, "split": PROMPT_SPLIT, "max_iter": MAX_ITER, "n_tasks": N_TASKS},
           "results": results}, open(BCB_OUT, "w"), ensure_ascii=False)

# Visualisation
plot_bcb(results, MAX_ITER, save_path="/kaggle/working/performance_bcb.png")   

In [ ]:
# ======================================== BIGCODEBENCH — MÉTRIQUES DÉTAILLÉES ========================================
import json, os, re, random, time, ast
from collections import Counter
from dataclasses import dataclass, field
import numpy as np
import matplotlib.pyplot as plt

# ----- CONFIG -----
SUBSET         = "hard"
PROMPT_SPLIT   = "instruct"
EXECUTION      = "gradio"
N_TASKS        = 50               # ← augmenté (20 → 50)
MAX_ITER       = 5
FEEDBACK       = False
REFORMULATE    = False
BCB_OUT        = "/kaggle/working/results_bcb.json"
WORK_DIR       = "/kaggle/working/bcb_work"

# ----- Task -----
@dataclass
class Task:
    task_id: str
    dataset: str
    prompt: str
    entry_point: str
    trusted_tests: list = field(default_factory=list)
    extra: dict = field(default_factory=dict)

# ----- Helpers -----
def extract_code(text: str) -> str:
    blocks = re.findall(r"```(?:python|py)?[ \t]*\n(.*?)```", text, re.S)
    if blocks:
        return max(blocks, key=len).strip()
    m = re.search(r"```(?:python|py)?[ \t]*\n(.*)", text, re.S)
    if m:
        return m.group(1).strip()
    return text.strip()

def syntax_error(code: str) -> str | None:
    try:
        ast.parse(code)
        return None
    except SyntaxError as e:
        return f"SyntaxError: {e.msg} (line {e.lineno})"

def has_infra_error(r: dict) -> bool:
    return r.get("intent") == "CODE" and r.get("final_error") == "SandboxInfraError"

# ----- solve -----
BUILDER_SYS = "Tu es un expert en programmation Python. Renvoie uniquement le code complet dans un bloc ```python."

def solve(llm, message, problem, max_iter, feedback, reformulate_on):
    t0 = time.time()
    trace = []
    prompt = reformulate(llm, message) if reformulate_on else message
    user_msg, code = prompt, ""
    for i in range(1, max_iter + 1):
        code = extract_code(llm.chat(BUILDER_SYS, user_msg, max_tokens=2048))
        err = syntax_error(code)
        trace.append({"iter": i, "ok": err is None, "error": err})
        if err is None:
            break
        user_msg = (f"{prompt}\n\nTon code précédent :\n```python\n{code}\n```\n"
                    f"a échoué avec {err}\nCorrige-le. Renvoie le code complet.")
    return {
        "intent": "CODE", "success": None, "final_code": code,
        "final_error": syntax_error(code), "iters": len(trace),
        "first_try_ok": len(trace) == 1 and syntax_error(code) is None,
        "trace": trace, "latency": time.time() - t0,
    }

# ----- 1) Chargement -----
def load_tasks(subset=SUBSET, split=PROMPT_SPLIT):
    from bigcodebench.data import get_bigcodebench
    key = "complete_prompt" if split == "complete" else "instruct_prompt"
    tasks = [Task(task_id=tid, dataset="bigcodebench", prompt=p[key],
                  entry_point=p.get("entry_point", "task_func"),
                  trusted_tests=[], extra={"libs": p.get("libs")})
             for tid, p in get_bigcodebench(subset=subset).items()]
    return sorted(tasks, key=lambda t: int(t.task_id.split("/")[1]))

# ----- 2) Scoring via le Space -----
def score(solutions: dict, work_dir: str = WORK_DIR, calibrated=True) -> dict:
    from bigcodebench.evaluate import evaluate
    from bigcodebench.data import get_bigcodebench
    os.makedirs(work_dir, exist_ok=True)
    samples_path = os.path.join(work_dir, "samples.jsonl")
    results_path = samples_path.replace(".jsonl", "_eval_results.json")

    def write_samples(items):
        with open(samples_path, "w") as f:
            for tid, code in items.items():
                f.write(json.dumps({"task_id": tid, "solution": code or "pass"}) + "\n")
        if os.path.isfile(results_path):
            os.remove(results_path)

    write_samples(solutions)
    try:
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples_path,
                 execution=EXECUTION, calibrated=calibrated,
                 selective_evaluate=",".join(str(int(t.split("/")[1])) for t in solutions))
    except Exception as e:
        print(f"selective_evaluate failed → retry full")
        all_ids = list(get_bigcodebench(subset=SUBSET))
        write_samples({tid: solutions.get(tid, "pass") for tid in all_ids})
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples_path,
                 execution=EXECUTION, calibrated=calibrated)

    if not os.path.isfile(results_path):
        raise RuntimeError(f"Pas de fichier résultats à {results_path}")
    data = json.load(open(results_path))["eval"]
    return {tid: bool(data.get(tid)) and data[tid][0]["status"] == "pass" for tid in solutions}

# ----- 3) Évaluation -----
def evaluate_bcb(llm, tasks, max_iter=3, feedback=False, reformulate=False):
    state = json.load(open(BCB_OUT)) if os.path.exists(BCB_OUT) else {}
    results = state.get("results", [])
    done = {r["task_id"] for r in results}

    for k, task in enumerate(tasks):
        if task.task_id in done:
            continue
        r = solve(llm, task.prompt, {"test": "", "entry_point": task.entry_point},
                  max_iter, feedback, reformulate)
        if has_infra_error(r):
            raise RuntimeError("Panne sandbox : " + str(r.get("trace")))
        r["task_id"] = task.task_id
        results.append(r)
        n_ok = sum(bool(x["success"]) for x in results if x["success"] is not None)
        print(f"[{k+1}/{len(tasks)}] {task.task_id}  iters={r['iters']}  "
              f"{r['latency']:.0f}s  | cumul : {n_ok}/{len(results)}")
        json.dump({"config": {"subset": SUBSET, "split": PROMPT_SPLIT, "max_iter": max_iter},
                   "results": results}, open(BCB_OUT, "w"), ensure_ascii=False)
    return results

# ----- 4) MÉTRIQUES DÉTAILLÉES -----
def compute_metrics(results: list) -> dict:
    """Calcule toutes les métriques du builder."""
    scored = [r for r in results if r["success"] is not None]
    n = len(scored)
    if n == 0:
        return {}

    n_ok = sum(r["success"] for r in scored)
    n_first = sum(r.get("first_try_ok", False) for r in scored)
    n_syntax_err = sum(r.get("final_error") is not None for r in scored if r.get("final_error"))
    latencies = [r["latency"] for r in scored]
    iters = [r["iters"] for r in scored]

    return {
        "n_tasks": n,
        "pass@1": n_ok / n,
        "pass@first_try": n_first / n,
        "pass@final": n_ok / n,
        "avg_iterations": np.mean(iters),
        "max_iterations": max(iters),
        "avg_latency_s": np.mean(latencies),
        "median_latency_s": np.median(latencies),
        "p95_latency_s": np.percentile(latencies, 95),
        "n_syntax_errors": n_syntax_err,
        "syntax_error_rate": n_syntax_err / n,
        "n_one_iter": sum(1 for i in iters if i == 1),
        "n_multi_iter": sum(1 for i in iters if i > 1),
        "multi_iter_rate": sum(1 for i in iters if i > 1) / n,
    }

# ----- 5) VISUALISATION COMPLÈTE (6 panneaux) -----
def plot_bcb(results: list, max_iter: int = 3, save_path: str = None):
    scored = [r for r in results if r["success"] is not None]
    n = len(scored)
    if n == 0:
        print("Pas de résultats scorés.")
        return

    m = compute_metrics(results)

    fig, ax = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle(f"BigCodeBench ({SUBSET}) — {n} tâches — pass@1 = {m['pass@1']*100:.1f}%",
                 fontsize=14, fontweight="bold")

    # --- (1) pass@1 vs pass@first_try ---
    ax[0,0].bar(["pass@1\n(final)", "pass@first_try"],
                [m["pass@1"]*100, m["pass@first_try"]*100],
                color=["#2171b5", "#6baed6"], width=0.5)
    ax[0,0].bar_label(ax[0,0].containers[0], fmt="%.1f%%", fontsize=12)
    ax[0,0].set_ylim(0, 105)
    ax[0,0].set_title("Taux de succès")

    # --- (2) Distribution des itérations ---
    iters = Counter(r["iters"] for r in scored)
    ks = list(range(1, max_iter + 1))
    colors = ["#31a354" if k == 1 else "#74c476" if k == 2 else "#fdae61" for k in ks]
    ax[0,1].bar([str(k) for k in ks], [iters.get(k, 0) for k in ks], color=colors)
    ax[0,1].bar_label(ax[0,1].containers[0], fontsize=11)
    ax[0,1].set_xlabel("Nombre d'itérations")
    ax[0,1].set_title(f"Itérations (moy. {m['avg_iterations']:.2f})")
    ax[0,1].annotate(f"1er essai : {m['n_one_iter']}/{n} ({m['n_one_iter']/n*100:.0f}%)",
                     xy=(0.05, 0.95), xycoords="axes fraction", fontsize=10,
                     bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

    # --- (3) Types d'erreurs (échecs) ---
    fails = scored if False else [r for r in scored if not r["success"]]
    if fails:
        # Classer par type
        err_types = []
        for r in fails:
            fe = r.get("final_error")
            if fe and "SyntaxError" in fe:
                err_types.append("SyntaxError")
            elif fe and "NameError" in fe:
                err_types.append("NameError")
            elif fe and "ImportError" in fe:
                err_types.append("ImportError")
            elif fe and "TypeError" in fe:
                err_types.append("TypeError")
            elif fe:
                err_types.append(fe.split(":")[0].strip()[:20])
            else:
                err_types.append("TestFailure")
        ec = Counter(err_types)
        items = ec.most_common(10)
        ax[0,2].barh([x[0] for x in items][::-1], [x[1] for x in items][::-1], color="#de2d26")
        ax[0,2].bar_label(ax[0,2].containers[0], fontsize=10)
    else:
        ax[0,2].text(0.5, 0.5, "Aucun échec", ha="center", va="center", fontsize=14)
    ax[0,2].set_title(f"Types d'erreurs ({len(fails)} échecs)")

    # --- (4) Latence par tâche (boxplot + points) ---
    latencies = [r["latency"] for r in scored]
    bp = ax[1,0].boxplot(latencies, patch_artist=True, boxprops=dict(facecolor="#a1dab4"))
    ax[1,0].set_ylabel("Latence (s)")
    ax[1,0].set_title(f"Latence (méd. {m['median_latency_s']:.0f}s, p95 {m['p95_latency_s']:.0f}s)")
    ax[1,0].annotate(f"moy. {m['avg_latency_s']:.0f}s", xy=(1.3, m["avg_latency_s"]),
                     fontsize=10, color="red")

    # --- (5) Cumul du score (courbe) ---
    cum = 0
    cum_vals = []
    for r in scored:
        cum += 1 if r["success"] else 0
        cum_vals.append(cum)
    ax[1,1].plot(range(1, n+1), cum_vals, "o-", color="#2171b5", markersize=4, linewidth=1.5)
    ax[1,1].axhline(m["pass@1"]*n, color="gray", linestyle="--", alpha=0.5,
                    label=f"moy. = {m['pass@1']*n:.1f}")
    ax[1,1].set_xlabel("Tâche")
    ax[1,1].set_ylabel("Score cumulé")
    ax[1,1].set_title("Progression du score")
    ax[1,1].legend()

    # --- (6) Tableau récapitulatif ---
    ax[1,2].axis("off")
    table_data = [
        ["Métrique", "Valeur"],
        ["Tâches testées", str(n)],
        ["pass@1 (final)", f"{m['pass@1']*100:.1f}%"],
        ["pass@first_try", f"{m['pass@first_try']*100:.1f}%"],
        ["Itérations moy.", f"{m['avg_iterations']:.2f}"],
        ["Multi-itérations", f"{m['n_multi_iter']}/{n} ({m['multi_iter_rate']*100:.0f}%)"],
        ["Latence moy.", f"{m['avg_latency_s']:.1f}s"],
        ["Latence médiane", f"{m['median_latency_s']:.1f}s"],
        ["Latence p95", f"{m['p95_latency_s']:.1f}s"],
        ["Erreurs de syntaxe", f"{m['n_syntax_errors']} ({m['syntax_error_rate']*100:.0f}%)"],
    ]
    table = ax[1,2].table(cellText=table_data[1:], colLabels=table_data[0],
                          loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.5)
    for i in range(len(table_data)):
        table[0, i].set_facecolor("#4393c3")
        table[0, i].set_text_props(color="white", fontweight="bold")
        if i > 0 and i % 2 == 0:
            for j in range(2):
                table[i, j].set_facecolor("#f0f0f0")
    ax[1,2].set_title("Récapitulatif")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    # --- Affichage console ---
    print("\n" + "="*50)
    print(f"  BigCodeBench ({SUBSET}) — {n} tâches")
    print("="*50)
    print(f"  pass@1 (final)      : {m['pass@1']*100:.1f}%  ({sum(r['success'] for r in scored)}/{n})")
    print(f"  pass@first_try      : {m['pass@first_try']*100:.1f}%  ({sum(r.get('first_try_ok', False) for r in scored)}/{n})")
    print(f"  Itérations moy.     : {m['avg_iterations']:.2f}")
    print(f"  Multi-itérations    : {m['n_multi_iter']}/{n} ({m['multi_iter_rate']*100:.0f}%)")
    print(f"  Latence moy.        : {m['avg_latency_s']:.1f}s")
    print(f"  Latence médiane     : {m['median_latency_s']:.1f}s")
    print(f"  Latence p95         : {m['p95_latency_s']:.1f}s")
    print(f"  Erreurs syntaxe     : {m['n_syntax_errors']} ({m['syntax_error_rate']*100:.0f}%)")
    print("="*50)

    # --- Détail par tâche ---
    print(f"\n{'Tâche':<22} {'OK':<4} {'Iters':<6} {'Lat(s)':<8} {'Erreur'}")
    print("-"*70)
    for r in scored:
        status = "✓" if r["success"] else "✗"
        err = (r.get("final_error") or "")[:40]
        print(f"{r['task_id']:<22} {status:<4} {r['iters']:<6} {r['latency']:<8.1f} {err}")

# ----- 6) Lancement -----
ALL_TASKS = load_tasks()
random.Random(0).shuffle(ALL_TASKS)
TASKS = ALL_TASKS[:N_TASKS]
print(f"{len(ALL_TASKS)} tâches au total, {len(TASKS)} sélectionnées")

llm = LLM()
results = evaluate_bcb(llm, TASKS, MAX_ITER, FEEDBACK, REFORMULATE)

# Scoring final via le Space
solutions = {r["task_id"]: r["final_code"] for r in results}
scores = score(solutions)
for r in results:
    r["success"] = scores.get(r["task_id"], False)

# Sauvegarde
json.dump({"config": {"subset": SUBSET, "split": PROMPT_SPLIT, "max_iter": MAX_ITER, "n_tasks": N_TASKS},
           "results": results}, open(BCB_OUT, "w"), ensure_ascii=False)

# Visualisation
plot_bcb(results, MAX_ITER, save_path="/kaggle/working/performance_bcb.png")   

In [ ]:
# ======================================== RECHARGEMENT + TABLEAU RÉCAPITULATIF ========================================
import json, os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

BCB_OUT = "/kaggle/working/results_bcb.json"

# ----- 1) Charger les résultats existants -----
data = json.load(open(BCB_OUT))
results = data["results"]
config = data.get("config", {})
print(f"Chargé {len(results)} résultats depuis {BCB_OUT}")
print(f"Config : {config}\n")

# ----- 2) Construire le DataFrame -----
rows = []
for r in results:
    # Feedback local pendant la boucle (ici = syntaxe OK ?)
    local_ok = None
    if r.get("trace"):
        local_ok = r["trace"][-1]["ok"]  # dernier essai local

    rows.append({
        "task_id": r["task_id"],
        "success_space": r.get("success", False),       # ✓/✗ au Space BigCodeBench
        "iters": r["iters"],                            # nombre d'itérations
        "first_try_ok": r.get("first_try_ok", False),   # réussi au 1er essai local ?
        "local_syntax_ok": local_ok,                    # syntaxe OK à la fin de la boucle ?
        "final_error": (r.get("final_error") or "")[:50],
        "latency_s": round(r.get("latency", 0), 1),
    })

df = pd.DataFrame(rows)

# ----- 3) Tableau principal -----
print("=" * 80)
print(f"  TABLEAU RÉCAPITULATIF — {len(df)} tâches  |  pass@1 = {df['success_space'].sum()}/{len(df)} "
      f"({df['success_space'].mean()*100:.1f}%)")
print("=" * 80)

# Tableau avec ✓/✗
df_display = df.copy()
df_display["success_space"] = df_display["success_space"].map({True: "✓ PASS", False: "✗ FAIL"})
df_display["local_syntax_ok"] = df_display["local_syntax_ok"].map({True: "✓", False: "✗", None: "?"})
df_display["first_try_ok"] = df_display["first_try_ok"].map({True: "✓", False: "✗"})
df_display["final_error"] = df_display["final_error"].replace("", "—")

print(df_display.to_string(index=False))

# ----- 4) Statistiques par itération -----
print("\n" + "=" * 60)
print("  RÉPARTITION PAR ITÉRATION")
print("=" * 60)

iter_stats = df.groupby("iters").agg(
    n_taches=("task_id", "count"),
    n_success=("success_space", "sum"),
).reset_index()
iter_stats["taux_succes"] = (iter_stats["n_success"] / iter_stats["n_taches"] * 100).round(1)
iter_stats.columns = ["itérations", "tâches", "succès", "taux_succes_%"]

print("\n" + iter_stats.to_string(index=False))

# ----- 5) Pourcentage "validé localement" vs "validé au Space" -----
print("\n" + "=" * 60)
print("  FEEDBACK LOCAL (syntaxe) vs SCORE SPACE")
print("=" * 60)

local_ok_mask = df["local_syntax_ok"] == True
space_ok_mask = df["success_space"] == True

n_local_ok = local_ok_mask.sum()
n_space_ok = space_ok_mask.sum()
n_both = (local_ok_mask & space_ok_mask).sum()
n_local_only = (local_ok_mask & ~space_ok_mask).sum()
n_space_only = (~local_ok_mask & space_ok_mask).sum()
n_neither = (~local_ok_mask & ~space_ok_mask).sum()

print(f"\n  {'Catégorie':<45} {'N':<6} {'%':<8}")
print(f"  {'─'*60}")
print(f"  {'Syntaxe OK + Space PASS (vraiment correct)':<45} {n_both:<6} {n_both/len(df)*100:<8.1f}%")
print(f"  {'Syntaxe OK + Space FAIL (bug logique)':<45} {n_local_only:<6} {n_local_only/len(df)*100:<8.1f}%")
print(f"  {'Syntaxe KO + Space PASS (rare, chance)':<45} {n_space_only:<6} {n_space_only/len(df)*100:<8.1f}%")
print(f"  {'Syntaxe KO + Space FAIL (code cassé)':<45} {n_neither:<6} {n_neither/len(df)*100:<8.1f}%")
print(f"  {'─'*60}")
print(f"  {'TOTAL':<45} {len(df):<6} 100.0%")

# ----- 6) Visualisation (3 panneaux) -----
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# (a) Pie : success vs fail
sizes = [n_space_ok, len(df) - n_space_ok]
colors = ["#31a354", "#de2d26"]
ax[0].pie(sizes, labels=[f"PASS ({n_space_ok})", f"FAIL ({len(df)-n_space_ok})"],
           colors=colors, autopct="%1.0f%%", startangle=90, textprops={"fontsize": 11})
ax[0].set_title(f"Succès au Space ({n_space_ok}/{len(df)})")

# (b) Barres : itérations vs succès
iters = df.groupby("iters")["success_space"].agg(["count", "sum"])
x = range(len(iters))
ax[1].bar([str(i) for i in iters.index], iters["count"], color="#a1dab4", label="Total")
ax[1].bar([str(i) for i in iters.index], iters["sum"], color="#31a354", label="Succès")
ax[1].set_xlabel("Itérations")
ax[1].set_ylabel("Tâches")
ax[1].set_title("Itérations : total vs succès")
ax[1].legend()
for i, (total, s) in enumerate(zip(iters["count"], iters["sum"])):
    ax[1].text(i, total + 0.2, f"{s}/{total}", ha="center", fontsize=10)

# (c) Barres horizontales : 4 catégories
cats = [
    ("Syntaxe ✓ + Space ✓", n_both, "#31a354"),
    ("Syntaxe ✓ + Space ✗", n_local_only, "#fdae61"),
    ("Syntaxe ✗ + Space ✓", n_space_only, "#74c476"),
    ("Syntaxe ✗ + Space ✗", n_neither, "#de2d26"),
]
y_pos = range(len(cats))
ax[2].barh([c[0] for c in cats][::-1], [c[1] for c in cats][::-1],
           color=[c[2] for c in cats][::-1])
ax[2].bar_label(ax[2].containers[0], fmt="%d", fontsize=11)
ax[2].set_title("Feedback local vs Space")
ax[2].set_xlabel("Tâches")

plt.tight_layout()
plt.savefig("/kaggle/working/recap_bcb.png", dpi=150, bbox_inches="tight")
plt.show()   

In [ ]:
# ======================================== BIGCODEBENCH — MÉTRIQUES DÉTAILLÉES ========================================
import json, os, re, random, time, ast
from collections import Counter
from dataclasses import dataclass, field
import numpy as np
import matplotlib.pyplot as plt

# ----- CONFIG -----
SUBSET         = "full"
PROMPT_SPLIT   = "instruct"
EXECUTION      = "gradio"
N_TASKS        = 50               # ← augmenté (20 → 50)
MAX_ITER       = 3
FEEDBACK       = False
REFORMULATE    = False
BCB_OUT        = "/kaggle/working/results_bcb.json"
WORK_DIR       = "/kaggle/working/bcb_work"

# ----- Task -----
@dataclass
class Task:
    task_id: str
    dataset: str
    prompt: str
    entry_point: str
    trusted_tests: list = field(default_factory=list)
    extra: dict = field(default_factory=dict)

# ----- Helpers -----
def extract_code(text: str) -> str:
    blocks = re.findall(r"```(?:python|py)?[ \t]*\n(.*?)```", text, re.S)
    if blocks:
        return max(blocks, key=len).strip()
    m = re.search(r"```(?:python|py)?[ \t]*\n(.*)", text, re.S)
    if m:
        return m.group(1).strip()
    return text.strip()

def syntax_error(code: str) -> str | None:
    try:
        ast.parse(code)
        return None
    except SyntaxError as e:
        return f"SyntaxError: {e.msg} (line {e.lineno})"

def has_infra_error(r: dict) -> bool:
    return r.get("intent") == "CODE" and r.get("final_error") == "SandboxInfraError"

# ----- solve -----
BUILDER_SYS = "Tu es un expert en programmation Python. Renvoie uniquement le code complet dans un bloc ```python."

def solve(llm, message, problem, max_iter, feedback, reformulate_on):
    t0 = time.time()
    trace = []
    prompt = reformulate(llm, message) if reformulate_on else message
    user_msg, code = prompt, ""
    for i in range(1, max_iter + 1):
        code = extract_code(llm.chat(BUILDER_SYS, user_msg, max_tokens=2048))
        err = syntax_error(code)
        trace.append({"iter": i, "ok": err is None, "error": err})
        if err is None:
            break
        user_msg = (f"{prompt}\n\nTon code précédent :\n```python\n{code}\n```\n"
                    f"a échoué avec {err}\nCorrige-le. Renvoie le code complet.")
    return {
        "intent": "CODE", "success": None, "final_code": code,
        "final_error": syntax_error(code), "iters": len(trace),
        "first_try_ok": len(trace) == 1 and syntax_error(code) is None,
        "trace": trace, "latency": time.time() - t0,
    }

# ----- 1) Chargement -----
def load_tasks(subset=SUBSET, split=PROMPT_SPLIT):
    from bigcodebench.data import get_bigcodebench
    key = "complete_prompt" if split == "complete" else "instruct_prompt"
    tasks = [Task(task_id=tid, dataset="bigcodebench", prompt=p[key],
                  entry_point=p.get("entry_point", "task_func"),
                  trusted_tests=[], extra={"libs": p.get("libs")})
             for tid, p in get_bigcodebench(subset=subset).items()]
    return sorted(tasks, key=lambda t: int(t.task_id.split("/")[1]))

# ----- 2) Scoring via le Space -----
def score(solutions: dict, work_dir: str = WORK_DIR, calibrated=True) -> dict:
    from bigcodebench.evaluate import evaluate
    from bigcodebench.data import get_bigcodebench
    os.makedirs(work_dir, exist_ok=True)
    samples_path = os.path.join(work_dir, "samples.jsonl")
    results_path = samples_path.replace(".jsonl", "_eval_results.json")

    def write_samples(items):
        with open(samples_path, "w") as f:
            for tid, code in items.items():
                f.write(json.dumps({"task_id": tid, "solution": code or "pass"}) + "\n")
        if os.path.isfile(results_path):
            os.remove(results_path)

    write_samples(solutions)
    try:
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples_path,
                 execution=EXECUTION, calibrated=calibrated,
                 selective_evaluate=",".join(str(int(t.split("/")[1])) for t in solutions))
    except Exception as e:
        print(f"selective_evaluate failed → retry full")
        all_ids = list(get_bigcodebench(subset=SUBSET))
        write_samples({tid: solutions.get(tid, "pass") for tid in all_ids})
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples_path,
                 execution=EXECUTION, calibrated=calibrated)

    if not os.path.isfile(results_path):
        raise RuntimeError(f"Pas de fichier résultats à {results_path}")
    data = json.load(open(results_path))["eval"]
    return {tid: bool(data.get(tid)) and data[tid][0]["status"] == "pass" for tid in solutions}

# ----- 3) Évaluation -----
def evaluate_bcb(llm, tasks, max_iter=3, feedback=False, reformulate=False):
    state = json.load(open(BCB_OUT)) if os.path.exists(BCB_OUT) else {}
    results = state.get("results", [])
    done = {r["task_id"] for r in results}

    for k, task in enumerate(tasks):
        if task.task_id in done:
            continue
        r = solve(llm, task.prompt, {"test": "", "entry_point": task.entry_point},
                  max_iter, feedback, reformulate)
        if has_infra_error(r):
            raise RuntimeError("Panne sandbox : " + str(r.get("trace")))
        r["task_id"] = task.task_id
        results.append(r)
        n_ok = sum(bool(x["success"]) for x in results if x["success"] is not None)
        print(f"[{k+1}/{len(tasks)}] {task.task_id}  iters={r['iters']}  "
              f"{r['latency']:.0f}s  | cumul : {n_ok}/{len(results)}")
        json.dump({"config": {"subset": SUBSET, "split": PROMPT_SPLIT, "max_iter": max_iter},
                   "results": results}, open(BCB_OUT, "w"), ensure_ascii=False)
    return results

# ----- 4) MÉTRIQUES DÉTAILLÉES -----
def compute_metrics(results: list) -> dict:
    """Calcule toutes les métriques du builder."""
    scored = [r for r in results if r["success"] is not None]
    n = len(scored)
    if n == 0:
        return {}

    n_ok = sum(r["success"] for r in scored)
    n_first = sum(r.get("first_try_ok", False) for r in scored)
    n_syntax_err = sum(r.get("final_error") is not None for r in scored if r.get("final_error"))
    latencies = [r["latency"] for r in scored]
    iters = [r["iters"] for r in scored]

    return {
        "n_tasks": n,
        "pass@1": n_ok / n,
        "pass@first_try": n_first / n,
        "pass@final": n_ok / n,
        "avg_iterations": np.mean(iters),
        "max_iterations": max(iters),
        "avg_latency_s": np.mean(latencies),
        "median_latency_s": np.median(latencies),
        "p95_latency_s": np.percentile(latencies, 95),
        "n_syntax_errors": n_syntax_err,
        "syntax_error_rate": n_syntax_err / n,
        "n_one_iter": sum(1 for i in iters if i == 1),
        "n_multi_iter": sum(1 for i in iters if i > 1),
        "multi_iter_rate": sum(1 for i in iters if i > 1) / n,
    }

# ----- 5) VISUALISATION COMPLÈTE (6 panneaux) -----
def plot_bcb(results: list, max_iter: int = 3, save_path: str = None):
    scored = [r for r in results if r["success"] is not None]
    n = len(scored)
    if n == 0:
        print("Pas de résultats scorés.")
        return

    m = compute_metrics(results)

    fig, ax = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle(f"BigCodeBench ({SUBSET}) — {n} tâches — pass@1 = {m['pass@1']*100:.1f}%",
                 fontsize=14, fontweight="bold")

    # --- (1) pass@1 vs pass@first_try ---
    ax[0,0].bar(["pass@1\n(final)", "pass@first_try"],
                [m["pass@1"]*100, m["pass@first_try"]*100],
                color=["#2171b5", "#6baed6"], width=0.5)
    ax[0,0].bar_label(ax[0,0].containers[0], fmt="%.1f%%", fontsize=12)
    ax[0,0].set_ylim(0, 105)
    ax[0,0].set_title("Taux de succès")

    # --- (2) Distribution des itérations ---
    iters = Counter(r["iters"] for r in scored)
    ks = list(range(1, max_iter + 1))
    colors = ["#31a354" if k == 1 else "#74c476" if k == 2 else "#fdae61" for k in ks]
    ax[0,1].bar([str(k) for k in ks], [iters.get(k, 0) for k in ks], color=colors)
    ax[0,1].bar_label(ax[0,1].containers[0], fontsize=11)
    ax[0,1].set_xlabel("Nombre d'itérations")
    ax[0,1].set_title(f"Itérations (moy. {m['avg_iterations']:.2f})")
    ax[0,1].annotate(f"1er essai : {m['n_one_iter']}/{n} ({m['n_one_iter']/n*100:.0f}%)",
                     xy=(0.05, 0.95), xycoords="axes fraction", fontsize=10,
                     bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

    # --- (3) Types d'erreurs (échecs) ---
    fails = scored if False else [r for r in scored if not r["success"]]
    if fails:
        # Classer par type
        err_types = []
        for r in fails:
            fe = r.get("final_error")
            if fe and "SyntaxError" in fe:
                err_types.append("SyntaxError")
            elif fe and "NameError" in fe:
                err_types.append("NameError")
            elif fe and "ImportError" in fe:
                err_types.append("ImportError")
            elif fe and "TypeError" in fe:
                err_types.append("TypeError")
            elif fe:
                err_types.append(fe.split(":")[0].strip()[:20])
            else:
                err_types.append("TestFailure")
        ec = Counter(err_types)
        items = ec.most_common(10)
        ax[0,2].barh([x[0] for x in items][::-1], [x[1] for x in items][::-1], color="#de2d26")
        ax[0,2].bar_label(ax[0,2].containers[0], fontsize=10)
    else:
        ax[0,2].text(0.5, 0.5, "Aucun échec", ha="center", va="center", fontsize=14)
    ax[0,2].set_title(f"Types d'erreurs ({len(fails)} échecs)")

    # --- (4) Latence par tâche (boxplot + points) ---
    latencies = [r["latency"] for r in scored]
    bp = ax[1,0].boxplot(latencies, patch_artist=True, boxprops=dict(facecolor="#a1dab4"))
    ax[1,0].set_ylabel("Latence (s)")
    ax[1,0].set_title(f"Latence (méd. {m['median_latency_s']:.0f}s, p95 {m['p95_latency_s']:.0f}s)")
    ax[1,0].annotate(f"moy. {m['avg_latency_s']:.0f}s", xy=(1.3, m["avg_latency_s"]),
                     fontsize=10, color="red")

    # --- (5) Cumul du score (courbe) ---
    cum = 0
    cum_vals = []
    for r in scored:
        cum += 1 if r["success"] else 0
        cum_vals.append(cum)
    ax[1,1].plot(range(1, n+1), cum_vals, "o-", color="#2171b5", markersize=4, linewidth=1.5)
    ax[1,1].axhline(m["pass@1"]*n, color="gray", linestyle="--", alpha=0.5,
                    label=f"moy. = {m['pass@1']*n:.1f}")
    ax[1,1].set_xlabel("Tâche")
    ax[1,1].set_ylabel("Score cumulé")
    ax[1,1].set_title("Progression du score")
    ax[1,1].legend()

    # --- (6) Tableau récapitulatif ---
    ax[1,2].axis("off")
    table_data = [
        ["Métrique", "Valeur"],
        ["Tâches testées", str(n)],
        ["pass@1 (final)", f"{m['pass@1']*100:.1f}%"],
        ["pass@first_try", f"{m['pass@first_try']*100:.1f}%"],
        ["Itérations moy.", f"{m['avg_iterations']:.2f}"],
        ["Multi-itérations", f"{m['n_multi_iter']}/{n} ({m['multi_iter_rate']*100:.0f}%)"],
        ["Latence moy.", f"{m['avg_latency_s']:.1f}s"],
        ["Latence médiane", f"{m['median_latency_s']:.1f}s"],
        ["Latence p95", f"{m['p95_latency_s']:.1f}s"],
        ["Erreurs de syntaxe", f"{m['n_syntax_errors']} ({m['syntax_error_rate']*100:.0f}%)"],
    ]
    table = ax[1,2].table(cellText=table_data[1:], colLabels=table_data[0],
                          loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.5)
    for i in range(len(table_data)):
        table[0, i].set_facecolor("#4393c3")
        table[0, i].set_text_props(color="white", fontweight="bold")
        if i > 0 and i % 2 == 0:
            for j in range(2):
                table[i, j].set_facecolor("#f0f0f0")
    ax[1,2].set_title("Récapitulatif")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    # --- Affichage console ---
    print("\n" + "="*50)
    print(f"  BigCodeBench ({SUBSET}) — {n} tâches")
    print("="*50)
    print(f"  pass@1 (final)      : {m['pass@1']*100:.1f}%  ({sum(r['success'] for r in scored)}/{n})")
    print(f"  pass@first_try      : {m['pass@first_try']*100:.1f}%  ({sum(r.get('first_try_ok', False) for r in scored)}/{n})")
    print(f"  Itérations moy.     : {m['avg_iterations']:.2f}")
    print(f"  Multi-itérations    : {m['n_multi_iter']}/{n} ({m['multi_iter_rate']*100:.0f}%)")
    print(f"  Latence moy.        : {m['avg_latency_s']:.1f}s")
    print(f"  Latence médiane     : {m['median_latency_s']:.1f}s")
    print(f"  Latence p95         : {m['p95_latency_s']:.1f}s")
    print(f"  Erreurs syntaxe     : {m['n_syntax_errors']} ({m['syntax_error_rate']*100:.0f}%)")
    print("="*50)

    # --- Détail par tâche ---
    print(f"\n{'Tâche':<22} {'OK':<4} {'Iters':<6} {'Lat(s)':<8} {'Erreur'}")
    print("-"*70)
    for r in scored:
        status = "✓" if r["success"] else "✗"
        err = (r.get("final_error") or "")[:40]
        print(f"{r['task_id']:<22} {status:<4} {r['iters']:<6} {r['latency']:<8.1f} {err}")

# ----- 6) Lancement -----
ALL_TASKS = load_tasks()
random.Random(0).shuffle(ALL_TASKS)
TASKS = ALL_TASKS[:N_TASKS]
print(f"{len(ALL_TASKS)} tâches au total, {len(TASKS)} sélectionnées")

llm = LLM()
results = evaluate_bcb(llm, TASKS, MAX_ITER, FEEDBACK, REFORMULATE)

# Scoring final via le Space
solutions = {r["task_id"]: r["final_code"] for r in results}
scores = score(solutions)
for r in results:
    r["success"] = scores.get(r["task_id"], False)

# Sauvegarde
json.dump({"config": {"subset": SUBSET, "split": PROMPT_SPLIT, "max_iter": MAX_ITER, "n_tasks": N_TASKS},
           "results": results}, open(BCB_OUT, "w"), ensure_ascii=False)

# Visualisation
plot_bcb(results, MAX_ITER, save_path="/kaggle/working/performance_bcb.png")   

In [ ]:
# ======================================== BIGCODEBENCH — JUDGE LLM (feedback par LLM, pas par tests) ========================================
import json, os, re, random, time, ast
from collections import Counter
from dataclasses import dataclass, field
import numpy as np
import matplotlib.pyplot as plt

# ----- CONFIG -----
SUBSET         = "hard"
PROMPT_SPLIT   = "instruct"
EXECUTION      = "gradio"
N_TASKS        = 50
MAX_ITER       = 4
BCB_OUT        = "/kaggle/working/results_bcb_judge.json"
WORK_DIR       = "/kaggle/working/bcb_work_judge"

# ----- Task -----
@dataclass
class Task:
    task_id: str
    dataset: str
    prompt: str
    entry_point: str
    trusted_tests: list = field(default_factory=list)
    extra: dict = field(default_factory=dict)

# ----- Helpers -----
def extract_code(text: str) -> str:
    blocks = re.findall(r"```(?:python|py)?[ \t]*\n(.*?)```", text, re.S)
    if blocks:
        return max(blocks, key=len).strip()
    m = re.search(r"```(?:python|py)?[ \t]*\n(.*)", text, re.S)
    if m:
        return m.group(1).strip()
    return text.strip()

def syntax_error(code: str) -> str | None:
    try:
        ast.parse(code)
        return None
    except SyntaxError as e:
        return f"SyntaxError: {e.msg} (line {e.lineno})"

# ----- PROMPTS -----
BUILDER_SYS = ("Tu es un expert en programmation Python. Renvoie UNIQUEMENT le code complet "
               "dans un bloc ```python. Pas d'explication.")

JUDGE_SYS = (
    "Tu es un juge technique expert en Python. On te donne une CONSIGNÉ et un CODE généré. "
    "Analyse le code et réponds STRICTEMENT au format JSON suivant (pas d'autre texte) :\n"
    '{"works": true/false, "reasons": ["raison 1", "raison 2", ...], "confidence": 0.0-1.0}\n\n'
    "Règles :\n"
    "- works=true SI le code résout correctement la consigne (logique, cas limites, types de retour).\n"
    "- works=false SI il y a un bug logique, un cas limite non géré, un mauvais algorithme, "
    "un mauvais type de retour, ou une erreur de syntaxe.\n"
    "- reasons : liste des problèmes concrets (lignes, variables, cas non gérés). "
    "Si works=true, reasons=[].\n"
    "- Ne sois PAS trop indulgent : vérifie les cas limites, les types, les valeurs de retour."
)

def judge_code(llm, prompt: str, code: str) -> dict:
    """
    Envoie prompt + code au judge LLM.
    Retourne {"works": bool, "reasons": list[str], "confidence": float}
    """
    user_msg = f"CONSIGNÉ :\n{prompt}\n\nCODE :\n```python\n{code}\n```"
    raw = llm.chat(JUDGE_SYS, user_msg, max_tokens=512)

    # Parser le JSON (tolérant aux balises)
    m = re.search(r"\{.*\}", raw, re.S)
    if not m:
        return {"works": False, "reasons": [f"Judge n'a pas renvoyé de JSON valide : {raw[:200]}"],
                "confidence": 0.0}
    try:
        d = json.loads(m.group(0))
        return {"works": bool(d.get("works", False)),
                "reasons": d.get("reasons", []),
                "confidence": float(d.get("confidence", 0.5))}
    except (json.JSONDecodeError, ValueError):
        return {"works": False, "reasons": [f"Parse JSON judge failed : {raw[:200]}"],
                "confidence": 0.0}

# ----- solve avec JUDGE (pas de run_sandbox) -----
def solve_judge(llm, task: Task, max_iter: int) -> dict:
    """
    Boucle : builder génère → judge évalue → feedback au builder → itère.
    """
    t0 = time.time()
    trace = []
    prompt = task.prompt
    user_msg = prompt
    code = ""

    for i in range(1, max_iter + 1):
        # 1) Builder génère
        code = extract_code(llm.chat(BUILDER_SYS, user_msg, max_tokens=2048))

        # 2) Vérification syntaxe rapide (circuit-breaker)
        syn_err = syntax_error(code)
        if syn_err:
            trace.append({"iter": i, "judge_works": False, "reasons": [syn_err], "confidence": 1.0})
            user_msg = (f"{prompt}\n\nTon code :\n```python\n{code}\n```\n"
                        f"ERREUR DE SYNTAXE : {syn_err}\nCorrige-le. Renvoie le code complet.")
            continue

        # 3) Judge évalue
        verdict = judge_code(llm, prompt, code)
        trace.append({"iter": i, "judge_works": verdict["works"],
                      "reasons": verdict["reasons"], "confidence": verdict["confidence"]})

        if verdict["works"]:
            break  # le juge a validé → on arrête

        # 4) Feedback au builder
        reasons_text = "\n".join(f"  - {r}" for r in verdict["reasons"][:5])
        user_msg = (f"{prompt}\n\nTon code :\n```python\n{code}\n```\n"
                    f"Le juge a trouvé CES PROBLÈMES :\n{reasons_text}\n"
                    f"Corrige TOUTES ces erreurs. Renvoie le code complet.")

    final_syn = syntax_error(code)
    return {
        "task_id": task.task_id,
        "intent": "CODE",
        "success": None,          # rempli par score()
        "final_code": code,
        "final_error": final_syn,
        "iters": len(trace),
        "first_try_ok": len(trace) == 1 and trace[0].get("judge_works", False),
        "judge_accepted": trace[-1].get("judge_works", False) if trace else False,
        "judge_confidence": trace[-1].get("confidence", 0) if trace else 0,
        "trace": trace,
        "latency": time.time() - t0,
    }

# ----- 1) Chargement des tâches -----
def load_tasks(subset=SUBSET, split=PROMPT_SPLIT):
    from bigcodebench.data import get_bigcodebench
    key = "complete_prompt" if split == "complete" else "instruct_prompt"
    tasks = [Task(task_id=tid, dataset="bigcodebench", prompt=p[key],
                  entry_point=p.get("entry_point", "task_func"),
                  trusted_tests=[], extra={"libs": p.get("libs")})
             for tid, p in get_bigcodebench(subset=subset).items()]
    return sorted(tasks, key=lambda t: int(t.task_id.split("/")[1]))

# ----- 2) Scoring via le Space (ground truth) -----
def score(solutions: dict, work_dir: str = WORK_DIR, calibrated=True) -> dict:
    from bigcodebench.evaluate import evaluate
    from bigcodebench.data import get_bigcodebench
    os.makedirs(work_dir, exist_ok=True)
    samples_path = os.path.join(work_dir, "samples.jsonl")
    results_path = samples_path.replace(".jsonl", "_eval_results.json")

    def write_samples(items):
        with open(samples_path, "w") as f:
            for tid, code in items.items():
                f.write(json.dumps({"task_id": tid, "solution": code or "pass"}) + "\n")
        if os.path.isfile(results_path):
            os.remove(results_path)

    write_samples(solutions)
    try:
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples_path,
                 execution=EXECUTION, calibrated=calibrated,
                 selective_evaluate=",".join(str(int(t.split("/")[1])) for t in solutions))
    except Exception as e:
        print(f"selective_evaluate failed → full retry")
        all_ids = list(get_bigcodebench(subset=SUBSET))
        write_samples({tid: solutions.get(tid, "pass") for tid in all_ids})
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples_path,
                 execution=EXECUTION, calibrated=calibrated)

    if not os.path.isfile(results_path):
        raise RuntimeError(f"Pas de résultats à {results_path}")
    data = json.load(open(results_path))["eval"]
    return {tid: bool(data.get(tid)) and data[tid][0]["status"] == "pass" for tid in solutions}

# ----- 3) Évaluation -----
def evaluate_bcb_judge(llm, tasks, max_iter=4):
    state = json.load(open(BCB_OUT)) if os.path.exists(BCB_OUT) else {}
    results = state.get("results", [])
    done = {r["task_id"] for r in results}

    for k, task in enumerate(tasks):
        if task.task_id in done:
            continue
        r = solve_judge(llm, task, max_iter)
        results.append(r)

        # Affichage en temps réel
        judge_ok = "✓" if r["judge_accepted"] else "✗"
        n_ok = sum(x.get("judge_accepted", False) for x in results)
        print(f"[{k+1}/{len(tasks)}] {task.task_id}  judge={judge_ok}  "
              f"iters={r['iters']}  conf={r['judge_confidence']:.2f}  "
              f"{r['latency']:.0f}s  | cumul judge : {n_ok}/{len(results)}")

        json.dump({"config": {"subset": SUBSET, "split": PROMPT_SPLIT, "max_iter": max_iter},
                   "results": results}, open(BCB_OUT, "w"), ensure_ascii=False)
    return results

# ----- 4) Visualisation (6 panneaux) -----
def plot_judge(results: list, max_iter: int = 4, save_path: str = None):
    scored = [r for r in results if r.get("success") is not None]
    n = len(scored)
    if n == 0:
        print("Pas de résultats scorés.")
        return

    n_space_ok = sum(r["success"] for r in scored)
    n_judge_ok = sum(r.get("judge_accepted", False) for r in scored)
    n_both = sum(r["success"] and r.get("judge_accepted", False) for r in scored)
    n_judge_only = sum((not r["success"]) and r.get("judge_accepted", False) for r in scored)
    n_space_only = sum(r["success"] and not r.get("judge_accepted", False) for r in scored)
    n_neither = sum((not r["success"]) and not r.get("judge_accepted", False) for r in scored)

    iters = [r["iters"] for r in scored]
    latencies = [r["latency"] for r in scored]
    confs = [r.get("judge_confidence", 0) for r in scored]

    fig, ax = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle(f"BigCodeBench ({SUBSET}) — Judge LLM — {n} tâches", fontsize=14, fontweight="bold")

    # (1) pass@1 Space vs Judge
    ax[0,0].bar(["Space\n(ground truth)", "Judge LLM"],
                [n_space_ok/n*100, n_judge_ok/n*100],
                color=["#2171b5", "#e6550d"], width=0.5)
    ax[0,0].bar_label(ax[0,0].containers[0], fmt="%.1f%%", fontsize=12)
    ax[0,0].set_ylim(0, 105)
    ax[0,0].set_title("pass@1 : Space vs Judge")

    # (2) Itérations
    iters_c = Counter(iters)
    ks = list(range(1, max_iter + 1))
    colors = ["#31a354" if k == 1 else "#74c476" if k == 2 else "#fdae61" if k == 3 else "#de2d26" for k in ks]
    ax[0,1].bar([str(k) for k in ks], [iters_c.get(k, 0) for k in ks], color=colors)
    ax[0,1].bar_label(ax[0,1].containers[0], fontsize=11)
    ax[0,1].set_xlabel("Itérations")
    ax[0,1].set_title(f"Itérations (moy. {np.mean(iters):.2f})")

    # (3) 4 catégories (Judge vs Space)
    cats = [
        ("Judge ✓ + Space ✓\n(vraiment correct)", n_both, "#31a354"),
        ("Judge ✓ + Space ✗\n(judge trop indulgent)", n_judge_only, "#fdae61"),
        ("Judge ✗ + Space ✓\n(judge trop strict)", n_space_only, "#74c476"),
        ("Judge ✗ + Space ✗\n(vraiment faux)", n_neither, "#de2d26"),
    ]
    ax[0,2].barh([c[0] for c in cats][::-1], [c[1] for c in cats][::-1],
                 color=[c[2] for c in cats][::-1])
    ax[0,2].bar_label(ax[0,2].containers[0], fontsize=10)
    ax[0,2].set_title("Cohérence Judge vs Space")

    # (4) Confiance du judge
    ax[1,0].hist(confs, bins=10, color="#6baed6", edgecolor="white")
    ax[1,0].axvline(np.mean(confs), color="red", linestyle="--", label=f"moy. {np.mean(confs):.2f}")
    ax[1,0].set_xlabel("Confiance du judge")
    ax[1,0].set_ylabel("Tâches")
    ax[1,0].set_title("Distribution de la confiance du judge")
    ax[1,0].legend()

    # (5) Latence
    bp = ax[1,1].boxplot(latencies, patch_artist=True, boxprops=dict(facecolor="#a1dab4"))
    ax[1,1].set_ylabel("Latence (s)")
    ax[1,1].set_title(f"Latence (méd. {np.median(latencies):.0f}s, p95 {np.percentile(latencies,95):.0f}s)")

    # (6) Tableau récap
    ax[1,2].axis("off")
    table_data = [
        ["Métrique", "Valeur"],
        ["Tâches", str(n)],
        ["pass@1 (Space)", f"{n_space_ok/n*100:.1f}% ({n_space_ok}/{n})"],
        ["Judge accepté", f"{n_judge_ok/n*100:.1f}% ({n_judge_ok}/{n})"],
        ["Cohérence (both ✓)", f"{n_both/n*100:.1f}%"],
        ["Itérations moy.", f"{np.mean(iters):.2f}"],
        ["Confiance moy. judge", f"{np.mean(confs):.2f}"],
        ["Latence moy.", f"{np.mean(latencies):.1f}s"],
    ]
    table = ax[1,2].table(cellText=table_data[1:], colLabels=table_data[0],
                          loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.5)
    for i in range(len(table_data)):
        table[0, i].set_facecolor("#4393c3")
        table[0, i].set_text_props(color="white", fontweight="bold")
        if i > 0 and i % 2 == 0:
            for j in range(2):
                table[i, j].set_facecolor("#f0f0f0")
    ax[1,2].set_title("Récapitulatif")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    # Console
    print("\n" + "="*60)
    print(f"  JUDGE LLM — {n} tâches")
    print("="*60)
    print(f"  pass@1 (Space)       : {n_space_ok/n*100:.1f}%  ({n_space_ok}/{n})")
    print(f"  Judge accepté        : {n_judge_ok/n*100:.1f}%  ({n_judge_ok}/{n})")
    print(f"  Cohérence (both ✓)   : {n_both/n*100:.1f}%")
    print(f"  Judge trop indulgent : {n_judge_only}")
    print(f"  Judge trop strict    : {n_space_only}")
    print(f"  Itérations moy.      : {np.mean(iters):.2f}")
    print(f"  Confiance moy.       : {np.mean(confs):.2f}")
    print("="*60)

# ----- 5) Lancement -----
ALL_TASKS = load_tasks()
random.Random(0).shuffle(ALL_TASKS)
TASKS = ALL_TASKS[:N_TASKS]
print(f"{len(ALL_TASKS)} tâches au total, {len(TASKS)} sélectionnées")

llm = LLM()
results = evaluate_bcb_judge(llm, TASKS, MAX_ITER)

# Scoring final via le Space (ground truth)
solutions = {r["task_id"]: r["final_code"] for r in results}
scores = score(solutions)
for r in results:
    r["success"] = scores.get(r["task_id"], False)

# Sauvegarde
json.dump({"config": {"subset": SUBSET, "split": PROMPT_SPLIT, "max_iter": MAX_ITER, "n_tasks": N_TASKS},
           "results": results}, open(BCB_OUT, "w"), ensure_ascii=False)

# Visualisation
plot_judge(results, MAX_ITER, save_path="/kaggle/working/performance_bcb_judge.png")   

In [ ]:
# ======================================== BIGCODEBENCH — JUDGE LLM (feedback par LLM, pas par tests) ========================================
import json, os, re, random, time, ast
from collections import Counter
from dataclasses import dataclass, field
import numpy as np
import matplotlib.pyplot as plt

# ----- CONFIG -----
SUBSET         = "hard"
PROMPT_SPLIT   = "instruct"
EXECUTION      = "gradio"
N_TASKS        = 50
MAX_ITER       = 6
BCB_OUT        = "/kaggle/working/results_bcb_judge.json"
WORK_DIR       = "/kaggle/working/bcb_work_judge"

# ----- Task -----
@dataclass
class Task:
    task_id: str
    dataset: str
    prompt: str
    entry_point: str
    trusted_tests: list = field(default_factory=list)
    extra: dict = field(default_factory=dict)

# ----- Helpers -----
def extract_code(text: str) -> str:
    blocks = re.findall(r"```(?:python|py)?[ \t]*\n(.*?)```", text, re.S)
    if blocks:
        return max(blocks, key=len).strip()
    m = re.search(r"```(?:python|py)?[ \t]*\n(.*)", text, re.S)
    if m:
        return m.group(1).strip()
    return text.strip()

def syntax_error(code: str) -> str | None:
    try:
        ast.parse(code)
        return None
    except SyntaxError as e:
        return f"SyntaxError: {e.msg} (line {e.lineno})"

# ----- PROMPTS -----
BUILDER_SYS = ("Tu es un expert en programmation Python. Renvoie UNIQUEMENT le code complet "
               "dans un bloc ```python. Pas d'explication.")

JUDGE_SYS = (
    "Tu es un juge technique expert en Python. On te donne une CONSIGNÉ et un CODE généré. "
    "Analyse le code et réponds STRICTEMENT au format JSON suivant (pas d'autre texte) :\n"
    '{"works": true/false, "reasons": ["raison 1", "raison 2", ...], "confidence": 0.0-1.0}\n\n'
    "Règles :\n"
    "- works=true SI le code résout correctement la consigne (logique, cas limites, types de retour).\n"
    "- works=false SI il y a un bug logique, un cas limite non géré, un mauvais algorithme, "
    "un mauvais type de retour, ou une erreur de syntaxe.\n"
    "- reasons : liste des problèmes concrets (lignes, variables, cas non gérés). "
    "Si works=true, reasons=[].\n"
    "- Ne sois PAS trop indulgent : vérifie les cas limites, les types, les valeurs de retour."
)

def judge_code(llm, prompt: str, code: str) -> dict:
    """
    Envoie prompt + code au judge LLM.
    Retourne {"works": bool, "reasons": list[str], "confidence": float}
    """
    user_msg = f"CONSIGNÉ :\n{prompt}\n\nCODE :\n```python\n{code}\n```"
    raw = llm.chat(JUDGE_SYS, user_msg, max_tokens=512)

    # Parser le JSON (tolérant aux balises)
    m = re.search(r"\{.*\}", raw, re.S)
    if not m:
        return {"works": False, "reasons": [f"Judge n'a pas renvoyé de JSON valide : {raw[:200]}"],
                "confidence": 0.0}
    try:
        d = json.loads(m.group(0))
        return {"works": bool(d.get("works", False)),
                "reasons": d.get("reasons", []),
                "confidence": float(d.get("confidence", 0.5))}
    except (json.JSONDecodeError, ValueError):
        return {"works": False, "reasons": [f"Parse JSON judge failed : {raw[:200]}"],
                "confidence": 0.0}

# ----- solve avec JUDGE (pas de run_sandbox) -----
def solve_judge(llm, task: Task, max_iter: int) -> dict:
    """
    Boucle : builder génère → judge évalue → feedback au builder → itère.
    """
    t0 = time.time()
    trace = []
    prompt = task.prompt
    user_msg = prompt
    code = ""

    for i in range(1, max_iter + 1):
        # 1) Builder génère
        code = extract_code(llm.chat(BUILDER_SYS, user_msg, max_tokens=2048))

        # 2) Vérification syntaxe rapide (circuit-breaker)
        syn_err = syntax_error(code)
        if syn_err:
            trace.append({"iter": i, "judge_works": False, "reasons": [syn_err], "confidence": 1.0})
            user_msg = (f"{prompt}\n\nTon code :\n```python\n{code}\n```\n"
                        f"ERREUR DE SYNTAXE : {syn_err}\nCorrige-le. Renvoie le code complet.")
            continue

        # 3) Judge évalue
        verdict = judge_code(llm, prompt, code)
        trace.append({"iter": i, "judge_works": verdict["works"],
                      "reasons": verdict["reasons"], "confidence": verdict["confidence"]})

        if verdict["works"]:
            break  # le juge a validé → on arrête

        # 4) Feedback au builder
        reasons_text = "\n".join(f"  - {r}" for r in verdict["reasons"][:5])
        user_msg = (f"{prompt}\n\nTon code :\n```python\n{code}\n```\n"
                    f"Le juge a trouvé CES PROBLÈMES :\n{reasons_text}\n"
                    f"Corrige TOUTES ces erreurs. Renvoie le code complet.")

    final_syn = syntax_error(code)
    return {
        "task_id": task.task_id,
        "intent": "CODE",
        "success": None,          # rempli par score()
        "final_code": code,
        "final_error": final_syn,
        "iters": len(trace),
        "first_try_ok": len(trace) == 1 and trace[0].get("judge_works", False),
        "judge_accepted": trace[-1].get("judge_works", False) if trace else False,
        "judge_confidence": trace[-1].get("confidence", 0) if trace else 0,
        "trace": trace,
        "latency": time.time() - t0,
    }

# ----- 1) Chargement des tâches -----
def load_tasks(subset=SUBSET, split=PROMPT_SPLIT):
    from bigcodebench.data import get_bigcodebench
    key = "complete_prompt" if split == "complete" else "instruct_prompt"
    tasks = [Task(task_id=tid, dataset="bigcodebench", prompt=p[key],
                  entry_point=p.get("entry_point", "task_func"),
                  trusted_tests=[], extra={"libs": p.get("libs")})
             for tid, p in get_bigcodebench(subset=subset).items()]
    return sorted(tasks, key=lambda t: int(t.task_id.split("/")[1]))

# ----- 2) Scoring via le Space (ground truth) -----
def score(solutions: dict, work_dir: str = WORK_DIR, calibrated=True) -> dict:
    from bigcodebench.evaluate import evaluate
    from bigcodebench.data import get_bigcodebench
    os.makedirs(work_dir, exist_ok=True)
    samples_path = os.path.join(work_dir, "samples.jsonl")
    results_path = samples_path.replace(".jsonl", "_eval_results.json")

    def write_samples(items):
        with open(samples_path, "w") as f:
            for tid, code in items.items():
                f.write(json.dumps({"task_id": tid, "solution": code or "pass"}) + "\n")
        if os.path.isfile(results_path):
            os.remove(results_path)

    write_samples(solutions)
    try:
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples_path,
                 execution=EXECUTION, calibrated=calibrated,
                 selective_evaluate=",".join(str(int(t.split("/")[1])) for t in solutions))
    except Exception as e:
        print(f"selective_evaluate failed → full retry")
        all_ids = list(get_bigcodebench(subset=SUBSET))
        write_samples({tid: solutions.get(tid, "pass") for tid in all_ids})
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples_path,
                 execution=EXECUTION, calibrated=calibrated)

    if not os.path.isfile(results_path):
        raise RuntimeError(f"Pas de résultats à {results_path}")
    data = json.load(open(results_path))["eval"]
    return {tid: bool(data.get(tid)) and data[tid][0]["status"] == "pass" for tid in solutions}

# ----- 3) Évaluation -----
def evaluate_bcb_judge(llm, tasks, max_iter=4):
    state = json.load(open(BCB_OUT)) if os.path.exists(BCB_OUT) else {}
    results = state.get("results", [])
    done = {r["task_id"] for r in results}

    for k, task in enumerate(tasks):
        if task.task_id in done:
            continue
        r = solve_judge(llm, task, max_iter)
        results.append(r)

        # Affichage en temps réel
        judge_ok = "✓" if r["judge_accepted"] else "✗"
        n_ok = sum(x.get("judge_accepted", False) for x in results)
        print(f"[{k+1}/{len(tasks)}] {task.task_id}  judge={judge_ok}  "
              f"iters={r['iters']}  conf={r['judge_confidence']:.2f}  "
              f"{r['latency']:.0f}s  | cumul judge : {n_ok}/{len(results)}")

        json.dump({"config": {"subset": SUBSET, "split": PROMPT_SPLIT, "max_iter": max_iter},
                   "results": results}, open(BCB_OUT, "w"), ensure_ascii=False)
    return results

# ----- 4) Visualisation (6 panneaux) -----
def plot_judge(results: list, max_iter: int = 4, save_path: str = None):
    scored = [r for r in results if r.get("success") is not None]
    n = len(scored)
    if n == 0:
        print("Pas de résultats scorés.")
        return

    n_space_ok = sum(r["success"] for r in scored)
    n_judge_ok = sum(r.get("judge_accepted", False) for r in scored)
    n_both = sum(r["success"] and r.get("judge_accepted", False) for r in scored)
    n_judge_only = sum((not r["success"]) and r.get("judge_accepted", False) for r in scored)
    n_space_only = sum(r["success"] and not r.get("judge_accepted", False) for r in scored)
    n_neither = sum((not r["success"]) and not r.get("judge_accepted", False) for r in scored)

    iters = [r["iters"] for r in scored]
    latencies = [r["latency"] for r in scored]
    confs = [r.get("judge_confidence", 0) for r in scored]

    fig, ax = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle(f"BigCodeBench ({SUBSET}) — Judge LLM — {n} tâches", fontsize=14, fontweight="bold")

    # (1) pass@1 Space vs Judge
    ax[0,0].bar(["Space\n(ground truth)", "Judge LLM"],
                [n_space_ok/n*100, n_judge_ok/n*100],
                color=["#2171b5", "#e6550d"], width=0.5)
    ax[0,0].bar_label(ax[0,0].containers[0], fmt="%.1f%%", fontsize=12)
    ax[0,0].set_ylim(0, 105)
    ax[0,0].set_title("pass@1 : Space vs Judge")

    # (2) Itérations
    iters_c = Counter(iters)
    ks = list(range(1, max_iter + 1))
    colors = ["#31a354" if k == 1 else "#74c476" if k == 2 else "#fdae61" if k == 3 else "#de2d26" for k in ks]
    ax[0,1].bar([str(k) for k in ks], [iters_c.get(k, 0) for k in ks], color=colors)
    ax[0,1].bar_label(ax[0,1].containers[0], fontsize=11)
    ax[0,1].set_xlabel("Itérations")
    ax[0,1].set_title(f"Itérations (moy. {np.mean(iters):.2f})")

    # (3) 4 catégories (Judge vs Space)
    cats = [
        ("Judge ✓ + Space ✓\n(vraiment correct)", n_both, "#31a354"),
        ("Judge ✓ + Space ✗\n(judge trop indulgent)", n_judge_only, "#fdae61"),
        ("Judge ✗ + Space ✓\n(judge trop strict)", n_space_only, "#74c476"),
        ("Judge ✗ + Space ✗\n(vraiment faux)", n_neither, "#de2d26"),
    ]
    ax[0,2].barh([c[0] for c in cats][::-1], [c[1] for c in cats][::-1],
                 color=[c[2] for c in cats][::-1])
    ax[0,2].bar_label(ax[0,2].containers[0], fontsize=10)
    ax[0,2].set_title("Cohérence Judge vs Space")

    # (4) Confiance du judge
    ax[1,0].hist(confs, bins=10, color="#6baed6", edgecolor="white")
    ax[1,0].axvline(np.mean(confs), color="red", linestyle="--", label=f"moy. {np.mean(confs):.2f}")
    ax[1,0].set_xlabel("Confiance du judge")
    ax[1,0].set_ylabel("Tâches")
    ax[1,0].set_title("Distribution de la confiance du judge")
    ax[1,0].legend()

    # (5) Latence
    bp = ax[1,1].boxplot(latencies, patch_artist=True, boxprops=dict(facecolor="#a1dab4"))
    ax[1,1].set_ylabel("Latence (s)")
    ax[1,1].set_title(f"Latence (méd. {np.median(latencies):.0f}s, p95 {np.percentile(latencies,95):.0f}s)")

    # (6) Tableau récap
    ax[1,2].axis("off")
    table_data = [
        ["Métrique", "Valeur"],
        ["Tâches", str(n)],
        ["pass@1 (Space)", f"{n_space_ok/n*100:.1f}% ({n_space_ok}/{n})"],
        ["Judge accepté", f"{n_judge_ok/n*100:.1f}% ({n_judge_ok}/{n})"],
        ["Cohérence (both ✓)", f"{n_both/n*100:.1f}%"],
        ["Itérations moy.", f"{np.mean(iters):.2f}"],
        ["Confiance moy. judge", f"{np.mean(confs):.2f}"],
        ["Latence moy.", f"{np.mean(latencies):.1f}s"],
    ]
    table = ax[1,2].table(cellText=table_data[1:], colLabels=table_data[0],
                          loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.5)
    for i in range(len(table_data)):
        table[0, i].set_facecolor("#4393c3")
        table[0, i].set_text_props(color="white", fontweight="bold")
        if i > 0 and i % 2 == 0:
            for j in range(2):
                table[i, j].set_facecolor("#f0f0f0")
    ax[1,2].set_title("Récapitulatif")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    # Console
    print("\n" + "="*60)
    print(f"  JUDGE LLM — {n} tâches")
    print("="*60)
    print(f"  pass@1 (Space)       : {n_space_ok/n*100:.1f}%  ({n_space_ok}/{n})")
    print(f"  Judge accepté        : {n_judge_ok/n*100:.1f}%  ({n_judge_ok}/{n})")
    print(f"  Cohérence (both ✓)   : {n_both/n*100:.1f}%")
    print(f"  Judge trop indulgent : {n_judge_only}")
    print(f"  Judge trop strict    : {n_space_only}")
    print(f"  Itérations moy.      : {np.mean(iters):.2f}")
    print(f"  Confiance moy.       : {np.mean(confs):.2f}")
    print("="*60)

# ----- 5) Lancement -----
ALL_TASKS = load_tasks()
random.Random(0).shuffle(ALL_TASKS)
TASKS = ALL_TASKS[:N_TASKS]
print(f"{len(ALL_TASKS)} tâches au total, {len(TASKS)} sélectionnées")

llm = LLM()
results = evaluate_bcb_judge(llm, TASKS, MAX_ITER)

# Scoring final via le Space (ground truth)
solutions = {r["task_id"]: r["final_code"] for r in results}
scores = score(solutions)
for r in results:
    r["success"] = scores.get(r["task_id"], False)

# Sauvegarde
json.dump({"config": {"subset": SUBSET, "split": PROMPT_SPLIT, "max_iter": MAX_ITER, "n_tasks": N_TASKS},
           "results": results}, open(BCB_OUT, "w"), ensure_ascii=False)

# Visualisation
plot_judge(results, MAX_ITER, save_path="/kaggle/working/performance_bcb_judge.png")   